# Number Theory Foundations for Cryptography
**Student:** Mahfuzur Rahman | **ID:** IT-23005  
**Course:** Cryptography and Cyber Law  
**Reference:** [IIT Bombay MNP Club - Number Theory & Cryptography Workshop](https://github.com/mnp-club/Number_Theory_and_Cryptography_Workshop)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IT23005/Cryptography-and-Cyber-Law/blob/main/CT-1%20Practicing%20Number%20Theory/01_Number_Theory_Foundations.ipynb)

---
## Overview
This notebook implements and validates foundational number theory algorithms essential for modern public-key cryptosystems:
1. **Extended Euclidean Algorithm** and Bézout's Identity: $d = \gcd(a, b) = a \cdot x + b \cdot y$
2. **Least Common Multiple (LCM)** via $\gcd(a, b) \cdot \operatorname{lcm}(a, b) = a \cdot b$
3. **Linear Diophantine Equations**: $a \cdot x + b \cdot y = c$
4. **Modular Multiplicative Inverse and Modular Division**: $b / a \pmod n$
5. **Fast Modular Exponentiation (Square-and-Multiply)**: $b^e \pmod m$ in $O(\log e)$
6. **Chinese Remainder Theorem (CRT)** constructive algorithm
7. **Repunits, Perfect Numbers, and Fermat Pseudoprimes**


### 1. Extended Euclidean Algorithm
Computes $d = \gcd(a, b)$ and integer coefficients $(x, y)$ such that $d = ax + by$.


In [ ]:
def extended_gcd(a, b):
    """
    Extended Euclidean Algorithm.
    Requires: a >= b >= 0 and a + b > 0.
    Returns: (d, x, y) such that d = gcd(a, b) = a*x + b*y.
    """
    assert a >= b and b >= 0 and a + b > 0, "Invalid inputs: requires a >= b >= 0 and a+b > 0"
    
    if b == 0:
        return (a, 1, 0)
        
    d, x1, y1 = extended_gcd(b, a % b)
    x = y1
    y = x1 - (a // b) * y1
    
    assert a % d == 0 and b % d == 0
    assert d == a * x + b * y
    return (d, x, y)

# Verification tests
print("extended_gcd(8, 4)  =", extended_gcd(8, 4))
print("extended_gcd(27, 15) =", extended_gcd(27, 15))


### 2. Least Common Multiple (LCM)
Uses the identity $\gcd(a, b) \cdot \operatorname{lcm}(a, b) = a \cdot b$.


In [ ]:
def lcm(a, b):
    """Returns the LCM of two positive integers."""
    assert a > 0 and b > 0, "Inputs must be positive integers"
    g = extended_gcd(max(a, b), min(a, b))[0]
    return (a * b) // g

assert lcm(9, 3) == 9
assert lcm(17, 37) == 629
print("lcm(9, 3) =", lcm(9, 3))
print("lcm(17, 37) =", lcm(17, 37))


### 3. Linear Diophantine Equations
Solves $a \cdot x + b \cdot y = c$. An integer solution exists if and only if $\gcd(a, b) \mid c$.


In [ ]:
def diophantine(a, b, c):
    """
    Finds integer solution (x, y) to a*x + b*y = c.
    """
    swapped = False
    if a < b:
        a_orig, b_orig = a, b
        a_val, b_val = b, a
        swapped = True
    else:
        a_orig, b_orig = a, b
        a_val, b_val = a, b
        
    gcd_val, x0, y0 = extended_gcd(a_val, b_val)
    assert c % gcd_val == 0, f"No solution: gcd({a_orig}, {b_orig}) does not divide {c}"
    
    factor = c // gcd_val
    x0 *= factor
    y0 *= factor
    
    x, y = (y0, x0) if swapped else (x0, y0)
    assert a_orig * x + b_orig * y == c
    return (x, y)

print("diophantine(34, 51, 170) =", diophantine(34, 51, 170))
print("diophantine(91, 117, 65)  =", diophantine(91, 117, 65))


### 4. Modular Multiplicative Inverse and Modular Division
Calculates $a^{-1} \pmod n$ such that $a \cdot a^{-1} \equiv 1 \pmod n$, and computes $b / a \pmod n$.


In [ ]:
def modInverse(a, n):
    """Calculates multiplicative inverse of a modulo n using extended Euclid."""
    assert n > 1
    a_reduced = a % n
    assert a_reduced > 0
    if a_reduced >= n:
        d, x, y = extended_gcd(a_reduced, n)
        inv = x % n
    else:
        d, x, y = extended_gcd(n, a_reduced)
        inv = y % n
    assert d == 1, f"Modular inverse does not exist: gcd({a}, {n}) != 1"
    return (inv + n) % n

def divide(b, a, n):
    """Returns x in [0, n-1] such that a * x = b (mod n)."""
    assert n > 1 and a > 0
    a_inv = modInverse(a, n)
    assert (a * a_inv) % n == 1
    return (a_inv * b) % n

print("divide(9, 5, 3)     =", divide(9, 5, 3))
print("divide(223, 67, 54) =", divide(223, 67, 54))


### 5. Fast Modular Exponentiation (Square-and-Multiply)
Computes $b^e \pmod m$ in $O(\log_2 e)$ modular multiplications, preventing integer overflow.


In [ ]:
def FastModularExponentiation(b, e, m):
    """Computes b^e mod m using Square-and-Multiply algorithm."""
    result = 1
    base = b % m
    exp = e
    while exp > 0:
        if exp % 2 == 1:
            result = (result * base) % m
        base = (base * base) % m
        exp //= 2
    return result

print("FastModularExponentiation(4, 5, 3) =", FastModularExponentiation(4, 5, 3))
print("Built-in pow(4, 5, 3)              =", pow(4, 5, 3))


### 6. Chinese Remainder Theorem (CRT)
Constructs the unique integer $r \pmod{n_1 n_2}$ satisfying:
$$r \equiv r_1 \pmod{n_1} \quad 	ext{and} \quad r \equiv r_2 \pmod{n_2}$$
where $\gcd(n_1, n_2) = 1$.


In [ ]:
def ChineseRemainderTheorem(n1, r1, n2, r2):
    """Constructs solution to system of congruences modulo n1*n2."""
    d, x, y = extended_gcd(n1, n2) if n1 >= n2 else extended_gcd(n2, n1)
    assert d == 1, "Moduli must be coprime"
    
    N = n1 * n2
    if n1 >= n2:
        # n1*x + n2*y = 1 => n2*y = 1 mod n1, n1*x = 1 mod n2
        r = (r1 * n2 * y + r2 * n1 * x) % N
    else:
        # n2*x + n1*y = 1 => n1*y = 1 mod n2, n2*x = 1 mod n1
        r = (r2 * n1 * y + r1 * n2 * x) % N
        
    r = (r + N) % N
    assert r % n1 == r1
    assert r % n2 == r2
    return r

print("ChineseRemainderTheorem(13, 7, 6, 5)  =", ChineseRemainderTheorem(13, 7, 6, 5))
print("ChineseRemainderTheorem(28, 20, 19, 3) =", ChineseRemainderTheorem(28, 20, 19, 3))


### 7. Repunits, Even Perfect Numbers & Fermat Pseudoprimes
Exploration of advanced theoretical properties from Day 2 of the workshop.


In [ ]:
# Repunits: R_n = 111...1 (n digits)
def repunit(n):
    return (10**n - 1) // 9

print("Repunits R_1 to R_6:", [repunit(i) for i in range(1, 7)])

# Even Perfect Numbers & Euclid-Euler Theorem
# n = 2^(p-1) * (2^p - 1) where (2^p - 1) is prime
def check_even_perfect(n):
    temp = n
    k = 0
    while temp % 2 == 0:
        temp //= 2
        k += 1
    p = k + 1
    m = (1 << p) - 1
    if temp == m:
        # check primality of m
        for i in range(2, int(m**0.5) + 1):
            if m % i == 0:
                return False, None
        return True, p
    return False, None

for num in [6, 28, 496, 8128]:
    is_p, p_val = check_even_perfect(num)
    print(f"Number {num}: Even Perfect = {is_p} (Mersenne prime exponent p = {p_val})")

# Fermat Pseudoprimes to base 2
# Composite n such that 2^(n-1) = 1 (mod n)
def is_pseudoprime_base2(n):
    if n < 4: return False
    # Check composite
    composite = any(n % i == 0 for i in range(2, int(n**0.5) + 1))
    if not composite: return False
    return FastModularExponentiation(2, n - 1, n) == 1

print("Is 341 a Fermat pseudoprime base 2?", is_pseudoprime_base2(341))
print("Factors of 341:", 11, "x", 31)
print("Is 561 a Carmichael number (pseudoprime to all coprime bases)?", is_pseudoprime_base2(561))
